# Modeling: sleep, part-time work and STAT2101 grade

This notebook fits the final model for the SleepWork-Grade project. The research question is whether weeknight sleep duration and weekly part-time work hours predict the final STAT2101 grade.

Model: ordinary least squares linear regression (scikit-learn `LinearRegression`) with the features `sleep_hours`, `work_hours` and `study_hours`. We evaluate it with leave-one-out cross-validation (LOO-CV) because n = 28 is small, and we fix the random seed to 2026. We chose a linear model over tree models because the sample is small and the coefficients are easy to interpret.

Run the EDA notebook first, and make sure the cleaned data file exists (`data/clean_student_sleep_survey_2026.csv`) before running this notebook.

In [ ]:
import os

# Make relative paths work whether Jupyter was started in the repo root or in notebooks/
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir(os.path.join(os.getcwd(), ".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import mean_squared_error, r2_score

from src.features import build_feature_matrix, add_sleep_deficit, add_work_heavy

SEED = 2026
np.random.seed(SEED)
print("numpy", np.__version__, "| pandas", pd.__version__)

## 1. Load the cleaned data

The cleaned file has 28 rows and 8 columns with no missing values. The feature matrix `X` and the target `y` come from the shared feature helper.

In [ ]:
df = pd.read_csv("data/clean_student_sleep_survey_2026.csv")
print(df.shape)
print(df.isna().sum().sum(), "missing values")

X, y = build_feature_matrix(df)
print("X:", X.shape, list(X.columns))
print("y:", y.shape, "mean =", round(y.mean(), 1))

## 2. Leave-one-out cross-validation

Each student is held out once. The model is trained on the other 27 rows and predicts the held-out grade. The RMSE and R2 below are computed on these out-of-sample predictions, so they are LOO-CV metrics, not training metrics.

In [ ]:
loo = LeaveOneOut()
model = LinearRegression()

y_pred = cross_val_predict(model, X, y, cv=loo)

rmse = np.sqrt(mean_squared_error(y, y_pred))
r2 = r2_score(y, y_pred)

print(f"Linear regression, LOO-CV (n = {len(y)})")
print(f"RMSE = {rmse:.2f}")
print(f"R2   = {r2:.2f}")

## 3. Baseline: predict the mean grade

To check whether the model adds anything, we compare it with a baseline that always predicts the mean grade of the training folds, using the same LOO-CV splits.

In [ ]:
dummy = DummyRegressor(strategy="mean")
base_pred = cross_val_predict(dummy, X, y, cv=loo)
base_rmse = np.sqrt(mean_squared_error(y, base_pred))

print(f"Mean baseline, LOO-CV RMSE = {base_rmse:.2f}")
print(f"Model improves RMSE by {base_rmse - rmse:.2f} grade points")

## 4. Final fit on all 28 rows

The LOO-CV metrics above estimate predictive performance. For interpretation, we refit the model on all 28 rows and report the coefficients.

In [ ]:
final_model = LinearRegression().fit(X, y)

coef_table = pd.DataFrame({
    "feature": X.columns,
    "coefficient": final_model.coef_.round(2),
})
print("Intercept:", round(final_model.intercept_, 1))
coef_table

## 5. Ablation: which features help?

We add features one at a time and compare LOO-CV RMSE and R2. The last row adds the engineered `sleep_deficit` and `work_heavy` features. These are kept only if they improve the LOO-CV score.

In [ ]:
def loo_metrics(data, cols):
    """Return LOO-CV RMSE and R2 for a linear model on the given columns."""
    preds = cross_val_predict(LinearRegression(), data[cols], y, cv=LeaveOneOut())
    return np.sqrt(mean_squared_error(y, preds)), r2_score(y, preds)

df_feat = add_work_heavy(add_sleep_deficit(df.copy()))

ablation_sets = {
    "sleep only": ["sleep_hours"],
    "sleep + work": ["sleep_hours", "work_hours"],
    "sleep + work + study": ["sleep_hours", "work_hours", "study_hours"],
    "sleep + work + study + deficit + work_heavy": [
        "sleep_hours", "work_hours", "study_hours", "sleep_deficit", "work_heavy"
    ],
}

rows = []
for name, cols in ablation_sets.items():
    r, s = loo_metrics(df_feat, cols)
    rows.append({"model": name, "LOO-CV RMSE": round(r, 2), "LOO-CV R2": round(s, 2)})

ablation = pd.DataFrame(rows)
ablation

The engineered features make the LOO-CV score worse, so the final model keeps only `sleep_hours`, `work_hours` and `study_hours`.

## 6. Predicted vs actual figure

This figure is the model-result figure for the final report. Points near the dashed 45-degree line are predicted well. The annotation shows the LOO-CV RMSE and R2.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 6.5))

ax.scatter(y, y_pred, s=50, alpha=0.85, edgecolor="black", linewidth=0.5, label="Student")

lo = min(y.min(), y_pred.min()) - 2
hi = max(y.max(), y_pred.max()) + 2
ax.plot([lo, hi], [lo, hi], linestyle="--", color="gray", label="Perfect prediction")
ax.set_xlim(lo, hi)
ax.set_ylim(lo, hi)

ax.set_xlabel("Actual course grade", fontsize=14)
ax.set_ylabel("Predicted course grade (LOO-CV)", fontsize=14)
ax.tick_params(axis="both", labelsize=12)
ax.set_title("Linear model: predicted vs actual", fontsize=16)
ax.text(0.05, 0.92, f"RMSE = {rmse:.2f}, R2 = {r2:.2f}",
        transform=ax.transAxes, fontsize=13,
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.9))
ax.legend(loc="lower right", fontsize=11)
ax.grid(alpha=0.3)

os.makedirs("report/figures", exist_ok=True)
fig.savefig("report/figures/predicted_vs_actual_grade.png", dpi=200, bbox_inches="tight")
plt.show()

## 7. Summary for the report

These are the values the Results section uses.

In [ ]:
print("Final model: linear regression on sleep_hours, work_hours, study_hours")
print(f"LOO-CV RMSE = {rmse:.2f} (mean baseline {base_rmse:.2f})")
print(f"LOO-CV R2   = {r2:.2f}")
print(f"Intercept   = {final_model.intercept_:.1f}")
for name, c in zip(X.columns, final_model.coef_):
    print(f"  {name:<12} {c:+.2f} grade points per unit")
print("Figure saved to report/figures/predicted_vs_actual_grade.png")